# Week 3: Анализ и визуализация данных

В этом ноутбуке используются три датасета из папки `data`:

- `countries_economy_area.csv` — данные о странах, площади, ВВП, ВВП на душу населения и валюте;
- `population.csv` — данные о населении стран за разные годы;
- `life.csv` — данные о продолжительности жизни по странам за разные годы.

Цель анализа — сравнить страны по экономическим и территориальным показателям, а также посмотреть динамику населения и продолжительности жизни по отдельным странам.

## 1. Импорт библиотек

In [ ]:
# подключаем библиотеки для таблиц и графиков
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# настраиваем общий вид графиков
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Загрузка данных из GitHub

In [ ]:
# указываем raw ссылки на csv файлы
economy_url = "https://raw.githubusercontent.com/vasilevsavelij135-spec/python-ai-Vasilev-Savelii/main/data/countries_economy_area.csv"
population_url = "https://raw.githubusercontent.com/vasilevsavelij135-spec/python-ai-Vasilev-Savelii/main/data/population.csv"
life_url = "https://raw.githubusercontent.com/vasilevsavelij135-spec/python-ai-Vasilev-Savelii/main/data/life.csv"

# читаем файлы в dataframe
df_economy = pd.read_csv(economy_url)
df_population = pd.read_csv(population_url)
df_life = pd.read_csv(life_url)

# смотрим размеры таблиц
print(f"economy: {df_economy.shape[0]} строк, {df_economy.shape[1]} столбцов")
print(f"population: {df_population.shape[0]} строк, {df_population.shape[1]} столбцов")
print(f"life: {df_life.shape[0]} строк, {df_life.shape[1]} столбцов")

# показываем первые строки
display(df_economy.head())
display(df_population.head())
display(df_life.head())

## 3. Подготовка данных

In [ ]:
# переименовываем столбцы в основной таблице
df_economy = df_economy.rename(columns={
    "country": "URL",
    "countryLabel": "country",
    "continentLabel": "continent",
    "capitalLabel": "capital",
    "area": "area_km2",
    "gdp": "gdp",
    "gdpPerCapita": "gdp_per_capita",
    "currencyLabel": "currency"
})

# переименовываем столбцы в таблице населения
df_population = df_population.rename(columns={
    "country": "URL",
    "countryLabel": "country",
    "population": "population",
    "date": "date"
})

# переименовываем столбцы в таблице продолжительности жизни
df_life = df_life.rename(columns={
    "country": "URL",
    "countryLabel": "country",
    "lifeExpectancy": "life_expectancy",
    "date": "date"
})

# приводим числовые столбцы к числам
for column in ["area_km2", "gdp", "gdp_per_capita"]:
    df_economy[column] = pd.to_numeric(df_economy[column], errors="coerce")

df_population["population"] = pd.to_numeric(df_population["population"], errors="coerce")
df_life["life_expectancy"] = pd.to_numeric(df_life["life_expectancy"], errors="coerce")

# приводим даты к типу datetime
df_population["date"] = pd.to_datetime(df_population["date"], errors="coerce", utc=True)
df_life["date"] = pd.to_datetime(df_life["date"], errors="coerce", utc=True)

# добавляем год для графиков динамики
df_population["year"] = df_population["date"].dt.year
df_life["year"] = df_life["date"].dt.year

# добавляем показатель ввп в триллионах для удобства чтения
df_economy["gdp_trillion"] = df_economy["gdp"] / 1_000_000_000_000

display(df_economy.head())
display(df_population.head())
display(df_life.head())

## 4. Проверка пропусков

In [ ]:
# проверяем пропуски в каждой таблице
print("пропуски в economy")
display(df_economy.isna().sum().reset_index().rename(columns={"index": "column", 0: "missing_count"}))

print("пропуски в population")
display(df_population.isna().sum().reset_index().rename(columns={"index": "column", 0: "missing_count"}))

print("пропуски в life")
display(df_life.isna().sum().reset_index().rename(columns={"index": "column", 0: "missing_count"}))

## График 1. Топ-10 стран по ВВП

Сначала сравним страны по общему ВВП. Этот показатель показывает масштаб экономики страны.

In [ ]:
# выбираем 10 стран с самым большим ввп
top_gdp = df_economy.sort_values("gdp", ascending=False).head(10)

# строим горизонтальную столбчатую диаграмму
plt.figure(figsize=(10, 6))
sns.barplot(data=top_gdp, x="gdp_trillion", y="country")

plt.title("Топ-10 стран по ВВП")
plt.xlabel("ВВП, трлн")
plt.ylabel("Страна")

plt.tight_layout()
plt.show()

**Вывод:**  
На графике видно, что общий ВВП сильнее всего сконцентрирован у крупнейших экономик мира. Лидерами являются страны с большим экономическим масштабом, но этот график не показывает уровень благополучия одного жителя.

## График 2. ВВП и площадь страны

Проверим, связана ли площадь страны с размером её экономики.

In [ ]:
# оставляем строки, где есть площадь и ввп
area_gdp = df_economy.dropna(subset=["area_km2", "gdp", "continent"])

# строим диаграмму рассеяния в логарифмической шкале
plt.figure(figsize=(11, 7))
sns.scatterplot(data=area_gdp, x="area_km2", y="gdp", hue="continent", size="gdp_per_capita", sizes=(30, 300), alpha=0.7)

plt.xscale("log")
plt.yscale("log")

plt.title("Связь площади страны и ВВП")
plt.xlabel("Площадь, км2")
plt.ylabel("ВВП")
plt.legend(title="Континент", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

**Вывод:**  
Большая площадь сама по себе не гарантирует высокий ВВП. На графике есть крупные по территории страны с разным экономическим уровнем. Это показывает, что территория является только одним из факторов, а не главным объяснением размера экономики.

## График 3. Средний ВВП на душу населения по континентам

ВВП на душу населения помогает сравнивать не размер экономики, а экономический показатель в пересчёте на одного человека.

In [ ]:
# считаем средний ввп на душу населения по континентам
continent_gdp_pc = (
    df_economy.dropna(subset=["continent", "gdp_per_capita"])
    .groupby("continent", as_index=False)["gdp_per_capita"]
    .mean()
    .sort_values("gdp_per_capita", ascending=False)
)

# строим диаграмму
plt.figure(figsize=(10, 6))
sns.barplot(data=continent_gdp_pc, x="gdp_per_capita", y="continent")

plt.title("Средний ВВП на душу населения по континентам")
plt.xlabel("Средний ВВП на душу населения")
plt.ylabel("Континент")

plt.tight_layout()
plt.show()

**Вывод:**  
Сравнение по континентам показывает, что экономические показатели распределены неравномерно. ВВП на душу населения лучше подходит для сравнения уровня экономического развития, чем общий ВВП.

## График 4. Динамика населения выбранных стран

Используем отдельный файл `population.csv`, чтобы посмотреть изменение населения во времени.

In [ ]:
# выбираем несколько стран для сравнения
selected_countries = ["США", "Китай", "Индия", "Япония", "Германия"]

# фильтруем данные по выбранным странам
population_selected = df_population[
    df_population["country"].isin(selected_countries) &
    df_population["year"].notna()
].copy()

# если названия пришли на английском, добавим запасной список
if population_selected.empty:
    selected_countries = ["United States", "China", "India", "Japan", "Germany"]
    population_selected = df_population[
        df_population["country"].isin(selected_countries) &
        df_population["year"].notna()
    ].copy()

# берём среднее значение за год, если за один год есть несколько записей
population_by_year = (
    population_selected
    .groupby(["country", "year"], as_index=False)["population"]
    .mean()
)

# строим линейный график
plt.figure(figsize=(11, 7))
sns.lineplot(data=population_by_year, x="year", y="population", hue="country", marker="o")

plt.title("Динамика населения выбранных стран")
plt.xlabel("Год")
plt.ylabel("Население")
plt.legend(title="Страна", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

**Вывод:**  
Данные по населению позволяют увидеть динамику во времени. У разных стран траектории отличаются: у одних население растёт быстрее, у других изменения более плавные.

## График 5. Динамика продолжительности жизни выбранных стран

Теперь используем файл `life.csv`, где собраны значения продолжительности жизни по годам.

In [ ]:
# выбираем те же страны для сравнения
selected_countries = ["США", "Китай", "Индия", "Япония", "Германия"]

# фильтруем данные по выбранным странам
life_selected = df_life[
    df_life["country"].isin(selected_countries) &
    df_life["year"].notna()
].copy()

# если названия пришли на английском, добавим запасной список
if life_selected.empty:
    selected_countries = ["United States", "China", "India", "Japan", "Germany"]
    life_selected = df_life[
        df_life["country"].isin(selected_countries) &
        df_life["year"].notna()
    ].copy()

# берём среднее значение за год, если за один год есть несколько записей
life_by_year = (
    life_selected
    .groupby(["country", "year"], as_index=False)["life_expectancy"]
    .mean()
)

# строим линейный график
plt.figure(figsize=(11, 7))
sns.lineplot(data=life_by_year, x="year", y="life_expectancy", hue="country", marker="o")

plt.title("Динамика продолжительности жизни выбранных стран")
plt.xlabel("Год")
plt.ylabel("Продолжительность жизни")
plt.legend(title="Страна", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

**Вывод:**  
График показывает, как менялась продолжительность жизни в разных странах. Этот показатель относится не к размеру экономики, а к социальному и демографическому развитию.

## График 6. ВВП на душу населения и последняя известная продолжительность жизни

Объединим основной экономический датасет с последними доступными значениями продолжительности жизни.

In [ ]:
# выбираем последние известные значения продолжительности жизни по каждой стране
latest_life = (
    df_life.dropna(subset=["date", "life_expectancy"])
    .sort_values("date")
    .groupby("URL", as_index=False)
    .tail(1)
)[["URL", "life_expectancy", "year"]]

# объединяем с основной таблицей
economy_life = df_economy.merge(latest_life, on="URL", how="inner")

# оставляем строки с нужными значениями
economy_life = economy_life.dropna(subset=["gdp_per_capita", "life_expectancy", "continent"])

# строим диаграмму рассеяния
plt.figure(figsize=(11, 7))
sns.scatterplot(data=economy_life, x="gdp_per_capita", y="life_expectancy", hue="continent", size="gdp", sizes=(30, 300), alpha=0.7)

plt.xscale("log")

plt.title("ВВП на душу населения и продолжительность жизни")
plt.xlabel("ВВП на душу населения")
plt.ylabel("Продолжительность жизни")
plt.legend(title="Континент", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

**Вывод:**  
На графике можно сравнить экономический показатель на одного человека с продолжительностью жизни. В целом более высокий ВВП на душу населения часто связан с более высокой продолжительностью жизни, но связь не является идеальной.

## Общий вывод

В этом ноутбуке были использованы три набора данных: основной экономический датасет, данные о населении и данные о продолжительности жизни.

Анализ показал, что общий ВВП отражает масштаб экономики страны, но не всегда показывает уровень жизни населения. Для сравнения стран между собой полезнее использовать ВВП на душу населения и социальные показатели, например продолжительность жизни.

Также видно, что площадь страны не объясняет экономический уровень напрямую. Большая территория может сочетаться как с высоким, так и с более умеренным ВВП.

Отдельные файлы `population.csv` и `life.csv` позволяют анализировать данные не только в разрезе стран, но и во времени. Это делает проект более содержательным, потому что появляется возможность изучать динамику населения и продолжительности жизни.

В целом выбранные данные подходят для анализа, так как содержат числовые и категориальные признаки, позволяют строить разные типы графиков и сравнивать страны по экономическим, территориальным и демографическим показателям.